# Pillar 3: FAERS pair-severity fraction

Target: severe outcomes among eligible submitted reports for a co-reported pair. It is distinct from DDI classification and does not establish causality. Existing model: ECFP6 + ExtraTrees with cold-drug splits. Next comparisons: single-drug expectation baseline, count-aware beta-binomial, and later-quarter validation if data exists.

In [ ]:
from pathlib import Path
import os, sys, subprocess

# Data preparation and audits run in Colab. Code comes from the GitHub checkout.
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
except ImportError:
    pass

DATA = Path("/content/drive/MyDrive/Projects/auditddi")
assert DATA.is_dir(), f"Google Drive project data folder not found: {DATA}"
REPO_VALUE = os.environ.get("AUDITDDI_CODE_ROOT", "").strip()
REPO_CANDIDATES = [Path(REPO_VALUE)] if REPO_VALUE else []
REPO_CANDIDATES += [Path.cwd(), Path("/content/auditddi"), Path("/content/auditddi_v2")]
REPO_CANDIDATES += list(Path.cwd().parents)
REPO = next((p.resolve() for p in REPO_CANDIDATES if (p / "src").is_dir() and (p / "requirements_colab.txt").is_file()), None)
if REPO is None:
    raise FileNotFoundError("GitHub checkout not found. Clone/pull https://github.com/akhilau-git/AuditDDI-project.git and set AUDITDDI_CODE_ROOT if needed.")
RESULTS_BASE = DATA / "results"
os.environ["AUDITDDI_DATA_BASE"] = str(DATA)
os.environ["AUDITDDI_RESULTS_BASE"] = str(RESULTS_BASE)
sys.path.insert(0, str(REPO / "src"))
RUNS = RESULTS_BASE / "pillars" / "drug_pair_severity"
RUNS.mkdir(parents=True, exist_ok=True)
print("Colab preparation environment; code:", REPO)
print("Drive data:", DATA)
print("Pillar outputs:", RUNS)

In [ ]:
%pip install -r {REPO / "requirements_colab.txt"}

In [ ]:
# Colab preparation check: confirm the audited FAERS pair table and its provenance.
PAIRS = DATA / "processed/seven_branch/faers_pair_severity_audit_2026q2.csv"
assert PAIRS.is_file(), f"Audited pair table not found: {PAIRS}"
import pandas as pd
frame = pd.read_csv(PAIRS, nrows=5)
print("Pair table:", PAIRS, "columns:", list(frame.columns))
print("Verify cohort period, pair canonicalization, report thresholds, and severe outcome counts before Kaggle training.")